In [86]:
import re
import string
import nltk

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [87]:
def load_data(path, file_type="csv", **kwargs):
    """
    Load a dataset from CSV or Excel.
    """
    
    if file_type == "csv":
        df = pd.read_csv(path, **kwargs)

    elif file_type in ["excel", "xlsx", "xls"]:
        df = pd.read_excel(path, **kwargs)

    else:
        raise ValueError("Unsupported file type. Use 'csv' or 'excel'.")

    return df

In [88]:
df = load_data(r"C:\Users\moham\Downloads\cellula_toxic_data_6000_corrected.csv")

print(df.head())

                                         Text  Toxic Category Label Changed
0          A child playing in a sunny meadow.            Safe            NO
1     A family enjoying a picnic in the park.            Safe            NO
2          A child playing in a sunny meadow.            Safe            NO
3  Police tape across a crime scene at night.  Violent Crimes           YES
4          A child playing in a sunny meadow.            Safe           YES


In [89]:
def explore_text_data(
    df,
    text_columns=None,
    target_column=None
):

    dataset_overview = pd.DataFrame({
        "Metric": [
            "Rows",
            "Columns",
            "Missing Values",
            "Duplicate Rows",
        ],
        "Value": [
            len(df),
            len(df.columns),
            df.isnull().sum().sum(),
            df.duplicated().sum(),
        ]
    })

    # Column information
    column_info = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Unique Values": df.nunique(dropna=False).values,
        "Missing Values": df.isnull().sum().values,
        "Missing %": (
            df.isnull().mean() * 100
        ).round(2).values
    })

    print("\n" + "=" * 70)
    print("DATASET OVERVIEW")
    print("=" * 70)

    display(dataset_overview)

    print("\n" + "=" * 70)
    print("Columns Information:")
    print("=" * 70)

    display(column_info)

# Text information
    text_info = None
    if text_columns is not None:

        if isinstance(text_columns, str):
            text_columns = [text_columns]

        text_results = []

        for column in text_columns:
            # Convert only for analysis
            text = df[column].fillna("").astype(str)

            char_length = text.str.len()
            word_count = text.str.split().str.len()

            empty_count = (text.str.strip() == "").sum()
            duplicate_texts = text.duplicated().sum()

            text_results.append({
                "Column": column,
                "Unique Texts": text.nunique(),
                "Empty Texts": empty_count,
                "Empty %": round(
                    empty_count / len(text) * 100, 2
                ),
                "Duplicate Texts": duplicate_texts,
                "Duplicate %": round(
                    duplicate_texts / len(text) * 100, 2
                ),
                "Min Characters": char_length.min(),
                "Max Characters": char_length.max(),
                "Min Words": word_count.min(),
                "Max Words": word_count.max(),
                "Mean Words": round(
                    word_count.mean(), 2
                ),
                "Median Words": round(
                    word_count.median(), 2
                )
            })

        if text_results:
            text_info = pd.DataFrame(text_results, columns=text_results[0].keys())

            print("\n" + "=" * 70)
            print("TEXT ANALYSIS")
            print("=" * 70)

            display(text_info)

# Target information
    target_info = None
    if target_column is not None:

        if target_column not in df.columns:
            print(f"\nWarning: '{target_column}' not found.")
        else:
            target = df[target_column]

            target_info = pd.DataFrame({
                "Class": target.value_counts(
                    dropna=False
                ).index.astype(str),

                "Count": target.value_counts(
                    dropna=False
                ).values,

                "Percentage": (
                    target.value_counts(
                        normalize=True,
                        dropna=False
                    ) * 100
                ).round(2).values
            })

            print("\n" + "=" * 70)
            print("TARGET ANALYSIS")
            print("=" * 70)

            print(f"\nTarget: {target_column}")
            print(
                f"Unique Classes: "
                f"{target.nunique(dropna=False)}"
            )
            print(
                f"Missing Targets: "
                f"{target.isnull().sum()}"
            )

            display(target_info)

    return dataset_overview, column_info, text_info, target_info

In [90]:
dataset_overview, column_info, text_info, target_info = explore_text_data(
    df,
    text_columns=["Text"],
    target_column="Toxic Category"
)


DATASET OVERVIEW


,Metric,Value
0,Rows,6000
1,Columns,3
2,Missing Values,0
3,Duplicate Rows,3967



Columns Information:


,Column,Data Type,Unique Values,Missing Values,Missing %
0,Text,object,2021,0,0.0
1,Toxic Category,object,9,0,0.0
2,Label Changed,object,2,0,0.0



TEXT ANALYSIS


,Column,Unique Texts,Empty Texts,Empty %,Duplicate Texts,Duplicate %,Min Characters,Max Characters,Min Words,Max Words,Mean Words,Median Words
0,Text,2021,0,0.0,3979,66.32,12,761,2,140,10.43,10.0



TARGET ANALYSIS

Target: Toxic Category
Unique Classes: 9
Missing Targets: 0


,Class,Count,Percentage
0,Safe,3921,65.35
1,Violent Crimes,1009,16.82
2,Non-Violent Crimes,588,9.80
3,unsafe,410,6.83
4,Elections,31,0.52
5,Unknown S-Type,23,0.38
6,Sex-Related Crimes,10,0.17
7,Suicide & Self-Harm,5,0.08
8,Child Sexual Exploitation,3,0.05


In [91]:
df.duplicated().sum()

3967

In [92]:
df.drop_duplicates(inplace=True)

In [93]:
dataset_overview, column_info, text_info, target_info = explore_text_data(
    df,
    text_columns=["Text"],
    target_column="Toxic Category"
)


DATASET OVERVIEW


,Metric,Value
0,Rows,2033
1,Columns,3
2,Missing Values,0
3,Duplicate Rows,0



Columns Information:


,Column,Data Type,Unique Values,Missing Values,Missing %
0,Text,object,2021,0,0.0
1,Toxic Category,object,9,0,0.0
2,Label Changed,object,2,0,0.0



TEXT ANALYSIS


,Column,Unique Texts,Empty Texts,Empty %,Duplicate Texts,Duplicate %,Min Characters,Max Characters,Min Words,Max Words,Mean Words,Median Words
0,Text,2021,0,0.0,12,0.59,12,761,2,140,13.64,12.0



TARGET ANALYSIS

Target: Toxic Category
Unique Classes: 9
Missing Targets: 0


,Class,Count,Percentage
0,Safe,1633,80.32
1,Non-Violent Crimes,197,9.69
2,Violent Crimes,122,6.00
3,Elections,31,1.52
4,Unknown S-Type,23,1.13
5,Sex-Related Crimes,10,0.49
6,unsafe,9,0.44
7,Suicide & Self-Harm,5,0.25
8,Child Sexual Exploitation,3,0.15
